<div style="color:#FF1F26;
           display:fill;
           border-style: solid;
           border-color:#C1C1C1;
           font-size:16px;
           font-family:Calibri;
           background-color:#B75351;">
<h2 style="text-align: center;
           padding: 10px;
           color:#FFFFFF;">
======= Model Interpretability - XGBoost + SHAP =======
</h2>
</div>



  <img src="https://www.hayleyfboyce.com/post/blog-01/featured_hucb4e3a8ff1f9aa3f10cb447f01c69f73_443478_680x500_fill_q90_lanczos_smart1.jpg" length="500" width="500">



<div style="padding:10px;
            color:white;
            margin:5;
            font-size:170%;
            text-align:left;
            display:fill;
            border-radius:15px;
            background-color:#294B8E;
            overflow:hidden;
            font-weight:700"> Table of Contents</div>

<a id="toc"></a>
- [1. About this Notebook](#1)
    - [1.1. Model Interpretability](#1.1)
        - [Global Importance](#1.1.1)    
        - [Local Importance](#1.1.2)    
    - [1.2. Data Set and Attributes](#1.2)
- [2. Setup](#2)
    - [2.1. Package and Library](#2.1)
    - [2.2. Load the Data](#2.2)
- [3. Explortory Data Analysis](#3)
    - [3.1. Distribution of Target Label](#3.1)
    - [3.2. Missing Value Handling / Replacement](#3.2)
    - [3.3. Correlation Analysis](#3.3)
    - [3.4. Histogram and Boxplot for Numerical Features](#3.4)
    - [3.5. Outlier Handling](#3.5)
    - [3.6. Count Plot for Categorical Features](#3.6)
- [4. Modelling](#4)
    - [4.1. XGBoost](#4.1)
- [5. Shapley Additive Explanations (Shap)](#5)
    - [5.1. Global Importance](#5.1)
    - [5.2. Local Importance](#5.2)
        - [5.2.1 Local Importance for Diagnosed Case](#5.2.1)
        - [5.2.2 Local Importance for Health Case](#5.2.2)


# <a id="1"></a>
<div style="padding:10px;
            color:white;
            margin:5;
            font-size:170%;
            text-align:left;
            display:fill;
            border-radius:15px;
            background-color:#294B8E;
            overflow:hidden;
            font-weight:700">1. About this Notebook</div>

This notebook demostrates how to use the most powerful boosting algorithm of XGBoost for prediction, while leveraging SHAP for explanning model inference.  Predictive modelling has been widely used today for informed decision making.  Besides making an accurate decisions, the model shall advise us the reasoning for the decisions. This not only improve transparaency of the model algorithm, but also stregenthen creditabiity of the decision.  


<a id="1.1"></a>
# <b><span style='color:#8D6D9B'>1.1. Model Interpretability</span></b>

Model interpretability in machine learning refers to the ability to explain the reasoning and decision-making process of a model in a way that is understandable to humans. It involves understanding how a model arrived at its predictions or classifications, and what factors and features it relied on to make those decisions.

Interpretability is particularly important in fields such as healthcare, finance, and legal systems, where decisions based on machine learning models can have significant consequences. It helps to build trust in the model, and can also be used to identify and address biases or errors in the model.

There are various techniques that can be used to enhance the interpretability of machine learning models, such as visualizations, feature importance analysis, and model-agnostic methods like LIME or SHAP values. The goal is to make the model transparent and explainable, while maintaining its predictive accuracy and usefulness.

<a id="1.1.1"></a>
### <b>Global Importance</span></b>

Global importance refers to the overall or aggregate impact of features on the model's predictions across the entire dataset or population. It provides a measure of how much each feature contributes to the model's performance or predictive power in a general sense, which helps in understanding the relative importance of features and can guide feature selection, feature engineering, and model interpretation. Global importance provides a high-level view of feature contributions to the model's predictions across the entire data distribution.

<a id="1.1.2"></a>
### <b>Local Importance</span></b>

Local importance, on the other hand, focuses on explaining the importance of features for individual predictions or instances. It aims to understand how the model arrived at a particular prediction by attributing the contribution of each feature to that specific prediction. Local importance provides explanations at the instance level, allowing for more fine-grained interpretation of model behavior. ocal importance can help identify the key features driving individual predictions, identify model biases or errors, and provide insights into the decision-making process of the model for specific instances.
<br><br>
In summary, global importance gives an overall understanding of feature importance and the model's behavior across the entire dataset, while local importance provides insights into individual predictions and helps in identifying specific feature contributions on a per-instance basis.

<a id="1.2"></a>
# <b><span style='color:#8D6D9B'>1.2. Data Set and Attributes</span></b>

The Kaggle Dataset [Heart Failure Prediction](https://www.kaggle.com/datasets/andrewmvd/heart-failure-clinical-data) is used for demostration in this notebook. The dataset consists of mortality by heart failure as target label, and there are 12 features that can be used to predict mortality by heart failure.

The data attributes are listed below. 


* <div style="font-size: 16px">DEATH_EVENT: target label [boolean]</div>
* <div style="font-size: 16px">Age: age of the patient [number]</div>
* <div style="font-size: 16px">Anaemia: Decrease of red blood cells or hemoglobin [categorical - M = male; F = female]</div>
* <div style="font-size: 16px">Creatinine_phosph: level of the CPK enzyme in the blood (mcg/L) [number]</div>
* <div style="font-size: 16px">Diabetes: the patient has diabetes or not [boolean]</div>
* <div style="font-size: 16px">Ejection_fraction: percentage of blood leaving the heart at each contraction (percentage) [number]</div>
* <div style="font-size: 16px">Gigh_blood_press: the patient has hypertension or not [boolean]</div>
* <div style="font-size: 16px">Platelets: platelets in the blood (kiloplatelets/mL) [number]</div>
* <div style="font-size: 16px">serum_creatinine: Level of serum creatinine in the blood (mg/dL) [number]</div>
* <div style="font-size: 16px">Serum_sodium: Level of serum sodium in the blood (mEq/L) [number]</div>
* <div style="font-size: 16px">Sex: gender [categorical]</div>
* <div style="font-size: 16px">Smoking: the patient smokes or not [boolean] </div>
* <div style="font-size: 16px">Time: Follow-up period (days) [number]</div>

# <a id="2"></a>
<div style="padding:10px;
            color:white;
            margin:5;
            font-size:170%;
            text-align:left;
            display:fill;
            border-radius:15px;
            background-color:#294B8E;
            overflow:hidden;
            font-weight:700">2. Setup</div>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

<a id="2.1"></a>
# <b><span style='color:#8D6D9B'>2.1. Package and Library</span></b>

In [ ]:
import warnings
from numba import NumbaDeprecationWarning
warnings.filterwarnings("ignore", category=NumbaDeprecationWarning)

# Data Standardization and Encoding
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modelling
from sklearn import model_selection, metrics
from sklearn.model_selection import train_test_split 

# Visualization Library, matplotlib and seaborn
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# Hide convergence warning for now
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Oversampling technique
from imblearn.over_sampling import SMOTE

# Random Forest
from sklearn.linear_model import LogisticRegression

import xgboost as xgb

# Random Search for Hyperparameter Turning
from sklearn.model_selection import RandomizedSearchCV

# Additional packages
from pandas.api.types import is_numeric_dtype
from scipy.stats import randint as sp_randint


# Model Explanation
import shap
from sklearn.inspection import permutation_importance



import random


<a id="2.2"></a>
# <b><span style='color:#8D6D9B'>2.2. Load the Data</span></b>

In [ ]:
# Read the data
df_heart = pd.read_csv('../input/heart-failure-clinical-data/heart_failure_clinical_records_dataset.csv')
print('No. of row: {}, no. of columns: {}'.format(df_heart.shape[0], df_heart.shape[1]))

# <a id="3"></a>
<div style="padding:10px;
            color:white;
            margin:5;
            font-size:170%;
            text-align:left;
            display:fill;
            border-radius:15px;
            background-color:#294B8E;
            overflow:hidden;
            font-weight:700">3. Explortory Data Analysis</div>

In [ ]:
# Basic information about the dataset
df_heart.info()

<a id="3.1"></a>
# <b><span style='color:#8D6D9B'>3.1. Distribution of Target Label</span></b>

In [ ]:
# check whether the data set is balanced

def auto_fmt (pct_value):
    return '{:.0f}\n({:.2f}%)'.format(df_heart['DEATH_EVENT'].value_counts().sum()*pct_value/100,pct_value) 

df_death_count = df_heart['DEATH_EVENT'].value_counts().rename_axis('Death Event').reset_index(name='Case Count')

fig = plt.gcf()
fig.set_size_inches(6,6)
plt.pie(x=df_death_count['Case Count'], labels=df_death_count['Death Event'], autopct=auto_fmt, textprops={'fontsize': 16})
plt.title('Distribution of Target Label (i.e. Death Event)',  fontsize = 16)

<div style=" background-color:#b22222;text-align:left; padding: 13px 13px; border-radius: 8px; color: white; font-size: 16px">
Observation: the distribution of target feature between death and non-death classes is not in equal proportion.  However, the objective of this notebook is for model explainability, and thus no oversampling will be performed for simplicity. 
</div>


<a id="3.2"></a>
# <b><span style='color:#8D6D9B'>3.2. Missing Value Handling / Replacement</span></b>

In [ ]:
df_null_value = df_heart.isnull().sum().rename_axis('Feature').reset_index(name='No of Null Value')

# Check if there are features with null value
df_null_value[df_null_value['No of Null Value']>0]

<div style=" background-color:#b22222;text-align:left; padding: 13px 13px; border-radius: 8px; color: white; font-size: 16px">
Observation: there is no missing value in the data set.  
</div>

<a id="3.3"></a>
# <b><span style='color:#8D6D9B'>3.3. Correlation Analysis</span></b>

In [ ]:
# Correlation matrix
corr_matrix = df_heart.corr()

fig, ax = plt.subplots(figsize=(10,10)) 
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

<div style=" background-color:#b22222;text-align:left; padding: 13px 13px; border-radius: 8px; color: white; font-size: 16px">
Observation: in general, the Heatmap shows low correlation among features, although the features of Smoking and Sex has correlation of 0.45, which shows a medium level of correlation. 
</div>

<a id="3.4"></a>
# <b><span style='color:#8D6D9B'>3.4. Histogram and Boxplot for Numerical Features</span></b>

In [ ]:

# Histogram of numerical features
numerical_features = ['age', 'creatinine_phosphokinase', 'ejection_fraction', 'platelets',
                      'serum_creatinine', 'serum_sodium', 'time']

# Histgram for numercial features
fig, ax = plt.subplots(4, 2, figsize=(16,20))

for i in range(0, len(numerical_features)):
    sns.histplot(data=df_heart, x=df_heart[numerical_features[i]], bins=25, ax=ax[int(i/2),i % 2])    

plt.show()


In [ ]:
# Box plot of numerical features
fig, ax = plt.subplots(4, 2, figsize=(16,20))

for i in range(0, len(numerical_features)):
    sns.boxplot(x="DEATH_EVENT",y=numerical_features[i],data=df_heart, ax=ax[int(i/2),i % 2])

plt.show()

<div style=" background-color:#b22222;text-align:left; padding: 13px 13px; border-radius: 8px; color: white; font-size: 16px">
Observation: there are outliers for numerical features of creatinine_phosphokinase, ejection_fraction, platelets, serum_creatinine and serum_sodium.  We will handle outliers with IRQ Method.  
</div>

<a id="3.5"></a>
# <b><span style='color:#8D6D9B'>3.5. Outlier Handling</span></b>

In [ ]:
for col in numerical_features:
    p75 = df_heart[df_heart[col] > 0][col].quantile(0.75)
    p25 = df_heart[df_heart[col] > 0][col].quantile(0.25)
    iqr = p75 - p25
    upper_limit = p75 + (1.5 * iqr)
    print('===={} with Upper Limit {:6.1f}, P75 {:6.1f}, P25 {:6.1f}, {} Outlier Records ========'.format(col, upper_limit, p75, p25, df_heart[df_heart[col] > upper_limit]['DEATH_EVENT'].count()))
    df_heart[col] = np.where (df_heart[col] > upper_limit, upper_limit, df_heart[col])

In [ ]:
# Re-print the Boxplot to check for outliers
fig, ax = plt.subplots(4, 2, figsize=(16,20))

for i in range(0, len(numerical_features)):
    sns.boxplot(x="DEATH_EVENT",y=numerical_features[i],data=df_heart, ax=ax[int(i/2),i % 2])

plt.show()

<div style=" background-color:#b22222;text-align:left; padding: 13px 13px; border-radius: 8px; color: white; font-size: 16px">
Observation: the IRQ is used to remove outliers. After the replacement of missing values and handling of abnormal values, the boxplots look more rational. 
</div>

<a id="3.6"></a>
# <b><span style='color:#8D6D9B'>3.6. Count Plot for Categorical Features</span></b>

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(16,12))


# Count plot of categorical features
categorical_features = ['anaemia', 'diabetes', 'high_blood_pressure', 'sex', 'smoking']
for feature, ax in zip(categorical_features, axs.flatten()):
    rel = sns.countplot(x=feature, data=df_heart, hue='DEATH_EVENT', ax=ax)
    rel.legend(['Normal', 'Death'])

<div style=" background-color:#b22222;text-align:left; padding: 13px 13px; border-radius: 8px; color: white; font-size: 16px">
Observation: from the count plots, it appears that there is an interdependence between a categorical variable and a target label.
</div>

# <a id="4"></a>
<div style="padding:10px;
            color:white;
            margin:5;
            font-size:170%;
            text-align:left;
            display:fill;
            border-radius:15px;
            background-color:#294B8E;
            overflow:hidden;
            font-weight:700">4. Modelling</div>

In [ ]:
# Split the data into train and test data
y = df_heart['DEATH_EVENT']
X = df_heart.drop(['DEATH_EVENT'], axis = 1)

# Splitting data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('No. of rows in X: {}, X_train: {}, and X_test: {}'.format(df_heart.shape[0], X_train.shape[0], X_test.shape[0]))

<a id="4.1"></a>
# <b><span style='color:#8D6D9B'>4.1. XGBoost</span></b>

In [ ]:
# XGBoost
model = xgb.XGBClassifier()
model.fit(X_train, y_train)


In [ ]:
# Prediction and accuracy score
y_pred = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)
print(metrics.classification_report(y_test, y_pred))

# <a id="5"></a>
<div style="padding:10px;
            color:white;
            margin:5;
            font-size:170%;
            text-align:left;
            display:fill;
            border-radius:15px;
            background-color:#294B8E;
            overflow:hidden;
            font-weight:700">5. Shapley Additive Explanations (Shap)</div>

<div style=" background-color:#515151;text-align:left; padding: 13px 13px; border-radius: 8px; color: white; font-size: 16px">
SHAP (SHapley Additive exPlanations) is a framework for model interpretability that calculates the impact of each feature on a prediction using Shapley values from cooperative game theory. It provides consistent and accurate explanations for individual predictions, allowing users to understand the importance of different features and detect potential biases in their machine learning models. By using SHAP, users can gain insights into their model's behavior in a flexible and efficient manner, fostering trust and understanding in machine learning applications.
</div>




<a id="5.1"></a>
# <b><span style='color:#8D6D9B'>5.1. Global Importance</span></b>

<div style=" background-color:#515151;text-align:left; padding: 13px 13px; border-radius: 8px; color: white; font-size: 16px">
Global Importance refers to understanding the overall impact and importance of features in the model across the entire dataset. It helps to identify the features that have the most significant influence on predictions in a general sense. 
<br><br>
The Global Importance of SHAP enable us to identify the most influential features in your model, understand their effects on predictions, and assess their relative importance in a global context.
</div>



In [ ]:
# Initialize the SHAP explainer
explainer = shap.Explainer(model, X)

# Calculate SHAP values for all instances
shap_values = explainer(X)

# Visualize global feature importance using summary plot
shap.summary_plot(shap_values, X)


<div style=" background-color:#b22222;text-align:left; padding: 13px 13px; border-radius: 8px; color: white; font-size: 16px">
Observation: Three important indication can be obtaned by analyzing the above SHAP chart.  First, the features at the top have the most significant impact on predictions, while those at the bottom have less influence. Next, the position of each feature bar indicates the range of its impact on predictions. Lastly, each dot represents a data point, and the density around a region represents the contration of feature values. 
<br><br>
Let's use the above results as examples, and the following findings can be derived.
<br><br>
- Feature Importance: Among the features analyzed, Time emerges as the most important feature, indicating that it has the most significant impact on predictions. Serum_creatinine follows as the second most important feature, while high_blood_pressure appears to have the least influence on predictions.
<br>
- Feature Effects: Features such as Time, Serum_creatinine, Ejection_fraction, Platelets, and Age exhibit a long tail to the right in the summary plot. This indicates that higher feature values for these variables have a greater impact on prediction. On the other hand, Time and Creatinine_phosphokinase show a tail to the left, suggesting that higher values of these features negatively impact the predictions.  It is noted that Time has both postive and negative impact on prediction.   
<br>
- Feature Value Density: Certain features, such as Time, Serum_creatinine, Ejection_fraction, and Diabetes, demonstrate a high density of data points within specific ranges of feature values. This suggests that there is a concentration of data points with feature values falling within those particular ranges.
<br><br>
Overall, the summary plot in SHAP helps you understand the relative importance of features in your model, their effects on predictions, and potential interactions between features. It assists in gaining insights into the decision-making process of your model and identifying areas for further analysis or improvement.    
</div>

<a id="5.2"></a>
# <b><span style='color:#8D6D9B'>5.2. Local Importance</span></b>

<div style=" background-color:#515151;text-align:left; padding: 13px 13px; border-radius: 8px; color: white; font-size: 16px">
Local Importance focuses on understanding the impact and contribution of features for individual predictions. It helps to explain why a specific prediction was made by highlighting the importance of different features for that particular instance.
<br><br>
The Local Importance of SHAP exhibits what features are crucial factors in prediction.
</div>


In [ ]:
# The SHAP library provides TreeExplaier for all tree-based algorithms, like LGBM and XGBoost
explainer = shap.TreeExplainer(model)

# Output the shap values of individual instances in a array format
shap_values = explainer.shap_values(X)

In [ ]:
# Setup the dataframe for the Shap values
df_shap = pd.DataFrame(shap_values, columns=X_test.columns)
df_shap.head(5)

In [ ]:
# For illustration, use all samples for prediction
y_pred = model.predict(X)

df_pred = pd.DataFrame(y_pred, columns=['pred'])
df_pred.head(5)

In [ ]:
# Check the shape of pred and shap dataframes
print('Pred size : {}, Shap size : {}'.format(df_pred.shape, df_shap.shape))

In [ ]:
# merge the prediction and shap values
df_pred_shap = pd.concat([df_pred,df_shap], axis=1)

df_pred_shap.head(5)

In [ ]:
# set the true and false prediction into two differnt dataframes
df_pred_shap_1 = df_pred_shap[df_pred_shap['pred'] == 1].reset_index()
df_pred_shap_0 = df_pred_shap[df_pred_shap['pred'] == 0].reset_index()

print('True : {}, False :{}'.format(df_pred_shap_1.shape, df_pred_shap_0.shape))

In [ ]:
# randomly select an index from a dataframe
def random_index_selection(dataframe):
    index_list = dataframe.index.tolist()
    random_index = random.choice(index_list)
    return random_index


<a id="5.2.1"></a>
# <b><span style='color:#8D6D9B'>5.2.1 Local Importance for Diagnosed Case</span></b>

In [ ]:
# randomly select 4 rows
idx = [None] * 4

for i in [0,1,2,3]:
    idx[i] = random_index_selection(df_pred_shap_1)

print(idx)

In [ ]:
# Manually select some instances with high variance for illustration purpose
idx =[67, 52, 79, 42]

# Select the row corresponding to instance 0
instances = df_pred_shap_1.drop(['pred','index'], axis=1)
# print(instance)

# Create a bar chart of the feature values
fig, ax = plt.subplots(2, 2, figsize=(15,10))

# Create a bar chart in the first subplot

ax[0, 0].bar(instances.iloc[idx[0],:].index.tolist(), instances.iloc[idx[0],:].values.tolist())
ax[0, 1].bar(instances.iloc[idx[1],:].index.tolist(), instances.iloc[idx[1],:].values.tolist())
ax[1, 0].bar(instances.iloc[idx[2],:].index.tolist(), instances.iloc[idx[2],:].values.tolist())
ax[1, 1].bar(instances.iloc[idx[3],:].index.tolist(), instances.iloc[idx[3],:].values.tolist())

# ax.bar(instance.index, instance.values, ax[0][0])

# # Set labels and title
# ax.set_xlabel('Feature')
# ax.set_ylabel('Value')

# Set title for the first subplot
ax[0, 0].set_title('SHAP Values of Instance ' + str(idx[0]))
ax[0, 1].set_title('SHAP Values of Instance ' + str(idx[1]))
ax[1, 0].set_title('SHAP Values of Instance ' + str(idx[2]))
ax[1, 1].set_title('SHAP Values of Instance ' + str(idx[3]))

# Rotate x-axis labels if needed
# Rotate x-axis labels if needed
ax[0, 0].tick_params(axis='x', rotation=90)
ax[0, 1].tick_params(axis='x', rotation=90)
ax[1, 0].tick_params(axis='x', rotation=90)
ax[1, 1].tick_params(axis='x', rotation=90)

# Show the plot
plt.show()

<div style=" background-color:#b22222;text-align:left; padding: 13px 13px; border-radius: 8px; color: white; font-size: 16px">
Observation: Based on the Global Importance of SHAP, Time, Serum_creatinine, and Ejection_fraction are identified as the top three features with the most significant impact on predictions. However, it is important to note that while the significance of these top features in the Global Importance holds true for most cases, there are variations observed in individual instances, as indicated by the Local Importance analysis.
<br><br>
For example, in instance 67, Time has a negative impact on the prediction, which deviates from the overall importance assigned to it in the Global Importance analysis. In instance 42, Time emerges as the sole predictor, while other features have minimal significance in determining the prediction outcome. Instances 52 and 79 also exhibit distinct distributions in the local importance assigned by SHAP, further emphasizing the variation in feature significance at the individual level.
<br><br>
In summary, Global Importance is essential for explaining which features are significant in terms of overall predictive power. However, when it comes to explaining the prediction outcome of individual instances, Local Importance assumes an even more crucial role in providing insights into the decision-making process.
    
</div>

<a id="5.2.2"></a>
# <b><span style='color:#8D6D9B'>5.2.2 Local Importance for Health Case</span></b>

In [ ]:
# randomly select 4 rows
idx = [None] * 4

for i in [0,1,2,3]:
    idx[i] = random_index_selection(df_pred_shap_0)

print(idx)

In [ ]:
# Manually select some instances with high variance for illustration purpose
idx =[6, 41, 133, 136]


# Select the row corresponding to instance 0
instances = df_pred_shap_0.drop(['pred','index'], axis=1)
# print(instance)

# Create a bar chart of the feature values
fig, ax = plt.subplots(2, 2, figsize=(15,10))

# Create a bar chart in the first subplot

ax[0, 0].bar(instances.iloc[idx[0],:].index.tolist(), instances.iloc[idx[0],:].values.tolist())
ax[0, 1].bar(instances.iloc[idx[1],:].index.tolist(), instances.iloc[idx[1],:].values.tolist())
ax[1, 0].bar(instances.iloc[idx[2],:].index.tolist(), instances.iloc[idx[2],:].values.tolist())
ax[1, 1].bar(instances.iloc[idx[3],:].index.tolist(), instances.iloc[idx[3],:].values.tolist())

# ax.bar(instance.index, instance.values, ax[0][0])

# # Set labels and title
# ax.set_xlabel('Feature')
# ax.set_ylabel('Value')

# Set title for the first subplot
ax[0, 0].set_title('SHAP Values of Instance ' + str(idx[0]))
ax[0, 1].set_title('SHAP Values of Instance ' + str(idx[1]))
ax[1, 0].set_title('SHAP Values of Instance ' + str(idx[2]))
ax[1, 1].set_title('SHAP Values of Instance ' + str(idx[3]))

# Rotate x-axis labels if needed
# Rotate x-axis labels if needed
ax[0, 0].tick_params(axis='x', rotation=90)
ax[0, 1].tick_params(axis='x', rotation=90)
ax[1, 0].tick_params(axis='x', rotation=90)
ax[1, 1].tick_params(axis='x', rotation=90)

# Show the plot
plt.show()

<div style=" background-color:#b22222;text-align:left; padding: 13px 13px; border-radius: 8px; color: white; font-size: 16px">
Observation: When comparing the Local Importance of diagnosed cases, instances 133 and 136 demonstrate typical alignment between Global and Local Importance. However, instance 6 presents an intriguing Local Importance pattern with a high positive SHAP value for the Time feature. Similarly, instance 41 showcases a contradictory observation with a high positive value for the Serum_creatinine feature.
<br><br>
In instances 133 and 136, the Local Importance analysis aligns with the Global Importance, indicating consistent feature significance in determining the predictions. However, instance 6 stands out with a notable Local Importance pattern, indicating that the Time feature has a significant positive impact on the prediction outcome for this particular case. Similarly, in instance 41, the high positive value of the Serum_creatinine feature contradicts the overall feature importance, suggesting a unique influence on the prediction for this instance.
<br><br>
These diverse observations highlight the value of examining Local Importance to gain insights into individual cases, where specific features may deviate from the overall trends established by Global Importance analysis.  
</div>

## Thanks for your time to read my notebook. If you like my work, please give me an "upvote" as appreciation. Happy Kaggling together.